In [21]:
import os
import json
import asyncio
from typing import List
from agents import function_tool, Agent, Runner
from exa_py import Exa
import dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput ,AsyncOpenAI
from agents.model_settings import ModelSettings
from pydantic import BaseModel , Field
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict

In [57]:
dotenv.load_dotenv()
EXA_API_KEY = os.environ.get("EXA_API_KEY")
if not EXA_API_KEY:
    raise RuntimeError("Missing EXA_API_KEY. Set it in your environment.")
exa = Exa(api_key=EXA_API_KEY)

In [15]:
@function_tool
def search_code_context(
    query: str,
    num_results: int = 3,
    max_chars_per_doc: int = 1200,
    summarize: bool = True
) -> str:
    """
    Searches code-related sources (GitHub, docs, Q&A) and returns compact context
    for coding questions.

    Args:
        query: Natural language query (e.g., "Express.js middleware for authentication").
        num_results: How many results to fetch (default 3).
        max_chars_per_doc: Max characters of content per result (default 1200).
        summarize: Ask Exa to include a short summary for each result.

    Returns:
        A JSON string with a list of {title, url, published_date, summary, text}.
        The agent will read and use this to craft an answer.
    """
    # Grab both metadata and a chunk of page contents so the LLM has real context
    resp = exa.search_and_contents(
        query=query,
        num_results=num_results,
        # Optional knobs: include_domains=[...], start_published_date="2024-01-01"
        contents={
            "max_characters": max_chars_per_doc,
            "include_html": False,
            "summary": summarize,
        },
    )

    out: List[dict] = []
    for r in resp.results:
        out.append({
            "title": getattr(r, "title", None),
            "url": getattr(r, "url", None),
            "published_date": getattr(r, "published_date", None),
            "summary": getattr(r, "summary", None) if hasattr(r, "summary") else None,
            "text": getattr(r, "text", None),
        })

    return json.dumps(out, ensure_ascii=False)

In [79]:
groq_api_key = os.getenv('GROQ_API_KEY')
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
gpt_model = OpenAIChatCompletionsModel(model="meta-llama/llama-4-maverick-17b-128e-instruct", openai_client=groq_client)

In [88]:
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
gpt_model = OpenAIChatCompletionsModel(model="mistralai/mistral-small-3.2-24b-instruct:free", openai_client=openrouter_client)

In [89]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[search_code_context],
    model=gpt_model,
    model_settings=ModelSettings(tool_choice="required"),
)

In [90]:
user_task = "Latest AI Agent frameworks in 2025"
result = await Runner.run(search_agent, input=user_task, max_turns=3)
print("\n=== Final Answer ===\n")
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export



=== Final Answer ===

### Summary of Latest AI Agent Frameworks in 2025

In 2025, AI agent frameworks have evolved significantly, offering various tools for businesses to automate tasks, enhance operations, and create new product categories. Key frameworks include LangChain, LangGraph, CrewAI, AutoGen, and OpenAI Agents SDK. LangChain is praised for its modularity, extensive ecosystem, and transparency in decision-making, though it can be heavy for small tasks. LangGraph extends LangChain with graph-based workflow visualization, aiding in debugging and scalability. CrewAI focuses on collaborative autonomy, allowing agents to delegate tasks and share updates, but requires careful configuration for reliability. AutoGen, developed by Microsoft Research, excels in conversational orchestration and multi-agent reasoning but is less suited for rapid production deployment. The OpenAI Agents SDK, released in March 2025, offers a minimalist approach with robust tracing and guardrails, making it

OPENAI_API_KEY is not set, skipping trace export


In [71]:
result.final_output

"LangChain, LangGraph, CrewAI, OpenAI SDK, and AutoGen are some of the top AI agent frameworks in 2025. \nLangChain is a framework that helps connect prompts, APIs, and reasoning steps into one logical flow, and it works seamlessly with models like OpenAI, Anthropic, or even local LLMs. \nLangGraph builds on LangChain but adds a visual layer, representing workflows as directed graphs, and CrewAI focuses on what it calls collaborative autonomy, where agents can delegate tasks, share updates, and coordinate results almost like a real team. \nAutoGen brings conversational orchestration, making agents debate and collaborate through dialogue, and OpenAI SDK provides a minimalist Python framework designed for creating multi-agent workflows with robust tracing and guardrails. \nThese frameworks and tools have their pros and cons, but they offer a wide range of possibilities for businesses looking to leverage intelligent automation to improve operational efficiency. \nWhen choosing an AI agent

In [91]:
HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model=gpt_model,
    output_type=WebSearchPlan,
)

In [92]:
message = "Latest AI Agent frameworks in 2025, Give List only"


result = await Runner.run(planner_agent, message)
print(result.final_output)

searches=[WebSearchItem(reason='To find the most recent and relevant AI agent frameworks that are expected to be prominent in 2025.', query='Latest AI agent frameworks 2025'), WebSearchItem(reason='To identify emerging trends and new developments in AI agent frameworks that might be significant by 2025.', query='Emerging AI agent frameworks trends 2025'), WebSearchItem(reason='To gather insights from expert opinions and predictions about which AI agent frameworks will be leading in 2025.', query='Expert predictions on AI agent frameworks 2025')]


OPENAI_API_KEY is not set, skipping trace export


In [93]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("basitab208@gmail.com") # Change this to your verified email
    to_email = To("22dcs097@charusat.edu.in") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [94]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model=gpt_model,
)

In [95]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model=gpt_model,
    output_type=ReportData,
)

In [96]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

In [97]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

In [99]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")

Starting research...
Planning searches...


OPENAI_API_KEY is not set, skipping trace export


Will perform 3 searches
Searching...


OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


Finished searching
Thinking about report...


OPENAI_API_KEY is not set, skipping trace export


Finished writing report
Writing email...


OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


Email sent
Hooray!


OPENAI_API_KEY is not set, skipping trace export
